In [1]:
import os
print(os.getcwd())

/Users/nchaparla/Documents/Capstone_Project


In [2]:
import pandas as pd
import numpy as np
import os

RAW_PATH = "01_data/raw/"
PROCESSED_PATH = "01_data/processed/"

print("Working directory:", os.getcwd())
print("Raw files found:", os.listdir(RAW_PATH))

Working directory: /Users/nchaparla/Documents/Capstone_Project
Raw files found: ['HIQ_L.xpt', '.DS_Store', 'BPXO_L.xpt', 'HUQ_L.xpt', 'DEMO_L.xpt', 'BMX_L.xpt', 'TCHOL_L.xpt', 'BPQ_L.xpt', 'DIQ_L.xpt', 'GHB_L.xpt']


In [3]:
demo  = pd.read_sas(os.path.join(RAW_PATH, "DEMO_L.xpt"),  format="xport", encoding="utf-8")
diq   = pd.read_sas(os.path.join(RAW_PATH, "DIQ_L.xpt"),   format="xport", encoding="utf-8")
bpq   = pd.read_sas(os.path.join(RAW_PATH, "BPQ_L.xpt"),   format="xport", encoding="utf-8")
huq   = pd.read_sas(os.path.join(RAW_PATH, "HUQ_L.xpt"),   format="xport", encoding="utf-8")
hiq   = pd.read_sas(os.path.join(RAW_PATH, "HIQ_L.xpt"),   format="xport", encoding="utf-8")
ghb   = pd.read_sas(os.path.join(RAW_PATH, "GHB_L.xpt"),   format="xport", encoding="utf-8")
tchol = pd.read_sas(os.path.join(RAW_PATH, "TCHOL_L.xpt"), format="xport", encoding="utf-8")
bpx   = pd.read_sas(os.path.join(RAW_PATH, "BPXO_L.xpt"),  format="xport", encoding="utf-8")
bmx   = pd.read_sas(os.path.join(RAW_PATH, "BMX_L.xpt"),   format="xport", encoding="utf-8")

files = {"DEMO":demo,"DIQ":diq,"BPQ":bpq,"HUQ":huq,
         "HIQ":hiq,"GHB":ghb,"TCHOL":tchol,"BPX":bpx,"BMX":bmx}
for name, f in files.items():
    print(f"{name}: {f.shape[0]:,} rows | {f.shape[1]} cols | SEQN: {'SEQN' in f.columns}")

DEMO: 11,933 rows | 27 cols | SEQN: True
DIQ: 11,744 rows | 9 cols | SEQN: True
BPQ: 8,501 rows | 6 cols | SEQN: True
HUQ: 11,933 rows | 6 cols | SEQN: True
HIQ: 11,933 rows | 11 cols | SEQN: True
GHB: 7,199 rows | 3 cols | SEQN: True
TCHOL: 8,068 rows | 4 cols | SEQN: True
BPX: 7,801 rows | 12 cols | SEQN: True
BMX: 8,860 rows | 22 cols | SEQN: True


In [4]:
df = demo.copy()
for name, file in [("DIQ",diq),("BPQ",bpq),("HUQ",huq),("HIQ",hiq),
                   ("GHB",ghb),("TCHOL",tchol),("BPX",bpx),("BMX",bmx)]:
    df = df.merge(file, on="SEQN", how="left")
    print(f"After {name}: {df.shape[0]:,} rows")

print(f"\nMerged shape: {df.shape}")

After DIQ: 11,933 rows
After BPQ: 11,933 rows
After HUQ: 11,933 rows
After HIQ: 11,933 rows
After GHB: 11,933 rows
After TCHOL: 11,933 rows
After BPX: 11,933 rows
After BMX: 11,933 rows

Merged shape: (11933, 92)


In [5]:
print(f"Before filters: {df.shape[0]:,}")
df = df[df["RIDAGEYR"] >= 20]
print(f"After age filter: {df.shape[0]:,}")
df = df[~(df["RIDEXPRG"] == 1)]
print(f"After pregnancy exclusion: {df.shape[0]:,}")
has_lab = (df["LBXGH"].notna() | df["BPXOSY1"].notna() | df["LBXTC"].notna())
df = df[has_lab]
print(f"After lab filter: {df.shape[0]:,}")

Before filters: 11,933
After age filter: 7,809
After pregnancy exclusion: 7,768
After lab filter: 6,004


In [6]:
columns_needed = {
    "SEQN":"id", "WTMEC2YR":"survey_weight",
    "SDMVPSU":"psu", "SDMVSTRA":"strata",
    "RIDAGEYR":"age", "INDFMPIR":"poverty_income_ratio",
    "DMDEDUC2":"education",
    "LBXGH":"hba1c", "BPXOSY1":"systolic_bp",
    "BPXODI1":"diastolic_bp", "LBXTC":"total_cholesterol",
    "BMXBMI":"bmi", "BMXWAIST":"waist_circumference",
    "HUQ010":"self_rated_health", "HUQ030":"usual_care_site",
    "HUQ055":"outpatient_visits", "HUQ042":"er_visits",
    "HUQ090":"overnight_hosp",
    "HIQ011":"has_insurance",
    "HIQ032A":"private_insurance",
    "HIQ032B":"medicare", "HIQ032D":"medicaid",
}
cols_to_keep = {k:v for k,v in columns_needed.items() if k in df.columns}
missing = [k for k in columns_needed if k not in df.columns]
df = df[list(cols_to_keep.keys())].rename(columns=cols_to_keep)
print(f"Columns kept: {df.shape[1]}")
print(f"Not found: {missing}")

Columns kept: 22
Not found: []


In [7]:
# Gender and race
demo_fix = demo[["SEQN","RIAGENDR","RIDRETH3"]].copy()
demo_fix["RIAGENDR"] = demo_fix["RIAGENDR"].map({1:"Male", 2:"Female"})
demo_fix["RIDRETH3"] = demo_fix["RIDRETH3"].map({
    1:"Mexican American", 2:"Other Hispanic",
    3:"Non-Hispanic White", 4:"Non-Hispanic Black",
    6:"Non-Hispanic Asian", 7:"Other/Multiracial"})
demo_fix = demo_fix.rename(columns={"SEQN":"id","RIAGENDR":"gender","RIDRETH3":"race_ethnicity"})

# Diagnosis columns
diq_fix = diq[["SEQN","DIQ010"]].copy()
diq_fix["DIQ010"] = diq_fix["DIQ010"].map({1.0:1, 2.0:0, 3.0:0, 7.0:np.nan, 9.0:np.nan})
diq_fix = diq_fix.rename(columns={"SEQN":"id","DIQ010":"told_diabetic"})

bpq_fix = bpq[["SEQN","BPQ020","BPQ080"]].copy()
bpq_fix["BPQ020"] = bpq_fix["BPQ020"].map({1.0:1, 2.0:0, 7.0:np.nan, 9.0:np.nan})
bpq_fix["BPQ080"] = bpq_fix["BPQ080"].map({1.0:1, 2.0:0, 7.0:np.nan, 9.0:np.nan})
bpq_fix = bpq_fix.rename(columns={"SEQN":"id","BPQ020":"told_hypertensive","BPQ080":"told_high_cholesterol"})

# Merge all fixes in
df = df.merge(demo_fix, on="id", how="left")
df = df.merge(diq_fix,  on="id", how="left")
df = df.merge(bpq_fix,  on="id", how="left")

print(f"Shape after fix merges: {df.shape}")
print(f"\nGender:\n{df['gender'].value_counts()}")
print(f"\nRace:\n{df['race_ethnicity'].value_counts()}")

Shape after fix merges: (6004, 27)

Gender:
gender
Female    3292
Male      2712
Name: count, dtype: int64

Race:
race_ethnicity
Non-Hispanic White    3545
Non-Hispanic Black     736
Other Hispanic         606
Mexican American       398
Other/Multiracial      384
Non-Hispanic Asian     335
Name: count, dtype: int64


In [8]:
# Continuous imputation
for col in ["hba1c","total_cholesterol","systolic_bp","diastolic_bp",
            "bmi","waist_circumference","poverty_income_ratio"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Categorical imputation
for col in ["education","usual_care_site","has_insurance"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

# Recode insurance
df["has_insurance"] = df["has_insurance"].map({1:1, 2:0})

# Fix outcome variables
df["er_visits"] = df["er_visits"].replace(99.0, np.nan)
df["er_visit_binary"] = (df["er_visits"] >= 2).astype(int)
df["overnight_hosp"] = df["overnight_hosp"].replace({7.0:np.nan, 9.0:np.nan})
df["hosp_binary"] = df["overnight_hosp"].map({1.0:1, 2.0:0})

# Drop missing outcomes
before = df.shape[0]
df = df.dropna(subset=["er_visit_binary","hosp_binary"])
print(f"Dropped {before - df.shape[0]} rows with missing outcomes")
print(f"Rows remaining: {df.shape[0]:,}")
print(f"\nER visit rate:        {df['er_visit_binary'].mean()*100:.1f}%")
print(f"Hospitalization rate: {df['hosp_binary'].mean()*100:.1f}%")

Dropped 12 rows with missing outcomes
Rows remaining: 5,992

ER visit rate:        19.7%
Hospitalization rate: 15.6%


In [9]:
output_path = "01_data/processed/nhanes_analytic_clean.csv"
df.to_csv(output_path, index=False)
print(f"✅ Clean dataset saved: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

✅ Clean dataset saved: 5,992 rows × 29 columns
Columns: ['id', 'survey_weight', 'psu', 'strata', 'age', 'poverty_income_ratio', 'education', 'hba1c', 'systolic_bp', 'diastolic_bp', 'total_cholesterol', 'bmi', 'waist_circumference', 'self_rated_health', 'usual_care_site', 'outpatient_visits', 'er_visits', 'overnight_hosp', 'has_insurance', 'private_insurance', 'medicare', 'medicaid', 'gender', 'race_ethnicity', 'told_diabetic', 'told_hypertensive', 'told_high_cholesterol', 'er_visit_binary', 'hosp_binary']


In [10]:
print("Age range:", df["age"].min(), "-", df["age"].max())
print("Total rows before outcome drop:", df.shape[0] + 12)

# Check how many have each lab value
print("\nHas HbA1c:", df["hba1c"].notna().sum())
print("Has systolic BP:", df["systolic_bp"].notna().sum())
print("Has cholesterol:", df["total_cholesterol"].notna().sum())

Age range: 20.0 - 80.0
Total rows before outcome drop: 6004

Has HbA1c: 5992
Has systolic BP: 5992
Has cholesterol: 5992
